In [ ]:
# 🤖 MCP Code Review Bot - Tutorial

## What You'll Build

An AI-powered code review system that:
1. Runs in **Docker** containers
2. Uses **MCP (Model Context Protocol)** for tool orchestration
3. Integrates with **GitHub Actions** for automated PR reviews
4. Performs linting, security scanning, and complexity analysis

## Architecture

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Pull Request   │────►│  GitHub Actions  │────►│  Docker Container│
│   Opened        │     │   Workflow       │     │  (MCP Server)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
                                                          │
                                                          ▼
                                                   ┌──────────────────┐
                                                   │  Analysis Tools  │
                                                   │  • flake8        │
                                                   │  • bandit        │
                                                   │  • pytest        │
                                                   │  • radon         │
                                                   └──────────────────┘
                                                          │
                                                          ▼
                                                   ┌──────────────────┐
                                                   │  AI Agent (LLM)  │
                                                   │  Reviews results │
                                                   └──────────────────┘
                                                          │
                                                          ▼
                                                   ┌──────────────────┐
                                                   │  PR Comment      │
                                                   │  Posted to GitHub│
                                                   └──────────────────┘
```

## Project Structure

```
mcp-code-review-bot/
├── mcp_code_review_server.py   # MCP server with code analysis tools
├── Dockerfile                   # Docker container definition
├── requirements.txt             # Python dependencies
├── .github/
│   └── workflows/
│       └── code-review.yml      # GitHub Actions workflow
└── MCP_Code_Review_Tutorial.ipynb  # This notebook
```

## Step 1: Understanding the MCP Server

The MCP server (`mcp_code_review_server.py`) provides these tools:

### Available Tools

| Tool | Purpose | Example Usage |
|------|---------|---------------|
| `get_changed_files()` | List files in PR | `get_changed_files()` |
| `read_file(path)` | Read file contents | `read_file("src/main.py")` |
| `run_linter(path)` | Run flake8 | `run_linter("src/main.py")` |
| `run_tests(path)` | Run pytest | `run_tests("tests/")` |
| `check_security(path)` | Run bandit scan | `check_security("src/")` |
| `analyze_complexity(path)` | Calculate complexity | `analyze_complexity("src/main.py")` |
| `post_review_comment()` | Format PR comment | `post_review_comment(file, line, comment)` |

### How It Works

1. **REPO_PATH**: The server looks for code at `/repo` (set via env var)
2. **Git Integration**: Uses `git diff` to find changed files
3. **Security**: Path validation prevents directory traversal
4. **Output**: Returns JSON or formatted text for AI consumption

## Step 2: Build and Test Locally

### 2.1 Build the Docker Image

```bash
# Navigate to the project directory
cd "/Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp-code-review-bot"

# Build the Docker image
docker build -t mcp-code-review .

# Verify image was created
docker images | grep mcp-code-review
```

### 2.2 Test the Container

```bash
# Set the repo path (use quotes for paths with spaces)
REPO_PATH="/Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp-code-review-bot"

# Run the container interactively
docker run -it --rm \
  -v "$REPO_PATH:/repo" \
  -e REPO_PATH=/repo \
  mcp-code-review /bin/bash

# Inside the container, test the tools
python -c "
import subprocess
result = subprocess.run(['flake8', '--version'], capture_output=True, text=True)
print('flake8:', result.stdout)
"

# Exit the container
exit
```

### 2.3 Test with MCP Inspector

```bash
# Set the repo path (use quotes for paths with spaces)
REPO_PATH="/Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp-code-review-bot"

# Run MCP Inspector to test the server
docker run -it --rm \
  -v "$REPO_PATH:/repo" \
  -e REPO_PATH=/repo \
  -p 6274:6274 \
  mcp-code-review

# In another terminal, run the inspector
npx @modelcontextprotocol/inspector \
  docker run -i --rm \
  -v "$REPO_PATH:/repo" \
  mcp-code-review
```

## Step 3: Test the Tools Locally

### 3.1 Create a Test Python File

```python
# test_sample.py (create this file with intentional issues)
def bad_function(x,y,z):
    """A function with style issues."""
    unused_var = 42
    if x==y and y==z:
        return True
    return False

def complex_function(n):
    """High complexity function."""
    if n == 0:
        return 0
    elif n == 1:
        return 1
    elif n == 2:
        return 2
    elif n == 3:
        return 3
    elif n == 4:
        return 4
    else:
        return n
```

### 3.2 Run Analysis Tools

```bash
# Set the repo path (use quotes for paths with spaces)
REPO_PATH="/Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp-code-review-bot"

# Run flake8
docker run --rm -v "$REPO_PATH:/repo" mcp-code-review \
  flake8 --max-line-length=100 /repo/test_sample.py

# Run bandit (security)
docker run --rm -v "$REPO_PATH:/repo" mcp-code-review \
  bandit -f txt /repo/test_sample.py

# Run complexity analysis
docker run --rm -v "$REPO_PATH:/repo" mcp-code-review \
  python -m radon cc -s /repo/test_sample.py
```

## Step 4: Set Up GitHub Actions

### 4.1 Create GitHub Repository

1. Create a new repository on GitHub
2. Push this code to it:

```bash
# Navigate to the project directory
cd "/Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp-code-review-bot"

# Initialize git repository
git init
git add .
git commit -m "Initial commit: MCP Code Review Bot"
git branch -M main
git remote add origin https://github.com/YOUR_USERNAME/mcp-code-review-bot.git
git push -u origin main
```

### 4.2 Configure Secrets

Go to **Settings → Secrets and variables → Actions** and add:

| Secret | Value | Required |
|--------|-------|----------|
| `GITHUB_TOKEN` | Automatically provided | Yes |
| `OPENAI_API_KEY` | Your OpenAI API key | Optional (for AI features) |

### 4.3 Test the Workflow

1. Create a new branch:
```bash
# Navigate to the project directory
cd "/Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp-code-review-bot"

# Create and switch to a new branch
git checkout -b test-pr
```

2. Add a Python file with some issues:
```python
# bad_code.py
def test(x,y):
    if x==y:
        print("equal")
    return x+y
```

3. Commit and push:
```bash
# Navigate to the project directory
cd "/Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp-code-review-bot"

# Add, commit, and push
git add bad_code.py
git commit -m "Add test file"
git push origin test-pr
```

4. Create a Pull Request on GitHub
5. Watch the Actions tab for the workflow to run
6. Check the PR comments for the review

## Step 5: Understanding the GitHub Actions Workflow

The workflow (`.github/workflows/code-review.yml`) does:

1. **Checkout**: Gets the PR code with full git history
2. **Build**: Creates the Docker image with MCP server
3. **Analyze**: Runs linting, security, and complexity checks
4. **Report**: Posts results as a PR comment

### Workflow Triggers

```yaml
on:
  pull_request:
    types: [opened, synchronize]
    paths:
      - '**.py'  # Only on Python file changes
```

### Permissions

```yaml
permissions:
  pull-requests: write  # Needed to post comments
  contents: read        # Needed to read code
```

## Step 6: Extend the System

### 6.1 Add Custom Rules

Edit `mcp_code_review_server.py` to add new tools:

```python
@mcp.tool()
def check_docstrings(file_path: str) -> str:
    """Check if functions have docstrings."""
    import ast
    
    full_path = Path(REPO_PATH) / file_path
    with open(full_path) as f:
        tree = ast.parse(f.read())
    
    missing = []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            if not ast.get_docstring(node):
                missing.append(node.name)
    
    if missing:
        return f"Functions missing docstrings: {', '.join(missing)}"
    return "✅ All functions have docstrings!"
```

### 6.2 Add AI Integration

To add AI-powered reviews, modify the workflow:

```python
# In run_review.py, add AI analysis:
import openai

def ai_review(file_content, lint_results):
    """Have AI review the code."""
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{
            "role": "system",
            "content": "You are a code reviewer. Analyze this code and suggest improvements."
        }, {
            "role": "user",
            "content": f"Code:\n{file_content}\n\nLint issues:\n{lint_results}"
        }]
    )
    return response.choices[0].message.content
```

### 6.3 Add More Tools

Install additional tools in the Dockerfile:

```dockerfile
RUN pip install --no-cache-dir \
    black \
    isort \
    mypy \
    pylint
```

Then add corresponding MCP tools.

## Step 7: Debugging

### Common Issues

| Issue | Solution |
|-------|----------|
| Docker build fails | Check Python version in base image |
| No files analyzed | Verify `paths: '**.py'` matches your files |
| Permission denied | Check GitHub token has PR write access |
| Tools not found | Ensure tools are installed in Dockerfile |
| Path errors with spaces | Use quotes around paths: `"$REPO_PATH:/repo"` |

### Local Debugging

```bash
# Set the repo path (use quotes for paths with spaces)
REPO_PATH="/Users/raymaldonado/Library/CloudStorage/GoogleDrive-vivachihuahua2004@gmail.com/My Drive/Code/oreilly-ai-agents/mcp-code-review-bot"

# Test Docker container locally
docker run -it --rm \
  -v "$REPO_PATH:/repo" \
  -e REPO_PATH=/repo \
  mcp-code-review /bin/bash

# Inside container, manually run tools
flake8 /repo/your_file.py
bandit -r /repo
pytest /repo/tests
```

### Viewing Logs

In GitHub Actions:
1. Go to Actions tab
2. Click on the workflow run
3. Expand the steps to see output

## Summary

You now have a working MCP-based code review system that:

✅ Runs in Docker containers
✅ Uses MCP protocol for tool orchestration
✅ Integrates with GitHub Actions
✅ Performs automated code analysis
✅ Posts reviews as PR comments

## Next Steps

1. **Customize rules**: Add your own linting preferences
2. **Add AI**: Integrate OpenAI/Claude for intelligent reviews
3. **Expand tools**: Add mypy, black, isort, etc.
4. **Notifications**: Add Slack/email notifications
5. **Metrics**: Track review statistics over time

## Resources

- [MCP Documentation](https://modelcontextprotocol.io/)
- [GitHub Actions Docs](https://docs.github.com/en/actions)
- [Docker Best Practices](https://docs.docker.com/develop/dev-best-practices/)